# Lab A - Evaluate agents with the Agent Development Kit (ADK)

**GENAI164 - Lab A of 2.** This lab builds an agent with the [Agent Development Kit (ADK)](https://adk.dev/) and evaluates it entirely with **ADK's own [evaluation framework](https://adk.dev/evaluate/)**, running locally in this notebook.

You build a simple customer service agent, test it, author eval data, and grade the agent with ADK's reference metrics and LLM-judge metrics.

This notebook runs in the **Gemini Enterprise Agent Platform Notebooks Workbench** environment in your lab project. Authentication is automatic through the instance service account.

> **Two-lab series.** Lab A (this notebook) covers build-time evaluation with ADK. **Lab B** is a separate notebook for the Gemini Enterprise Agent Platform managed eval tools (user simulation, custom and rubric metrics, and managed evaluation runs on a deployed agent). You do not deploy to or test on the platform in this lab.


## Lab overview

This notebook is organized into five parts:

- **Part 0: Set up your environment.** Install the Agent Development Kit and connect the notebook to your Google Cloud project. *Key ADK feature: the ADK toolkit with its built-in evaluation tools.*

- **Part 1: Build a simple ADK agent.** Create a small customer service agent that handles orders, refunds, and product questions, and chat with it once to confirm it works. *Key ADK feature: assembling an agent from a model, instructions, and tools, and running it in a local session.*

- **Part 2: Evaluate the agent locally with ADK.** Capture how the agent should behave as test data, then automatically grade a real run against it. *Key ADK feature: built-in evaluation that checks whether the agent took the right steps and gave the right answer, scored both by exact match and by an LLM acting as a judge.*

- **Part 3: Evaluate with ADK user simulation.** Let a simulated customer hold a free-flowing, multi-turn conversation with the agent, then score how it responded. *Key ADK feature: the user simulator that role-plays a customer toward a goal, so you can test multi-turn behavior and grade it for safety and hallucinations.*

- **Part 4: Optimize and verify.** Watch a weaker version of the agent fail an evaluation, fix its instruction, and confirm the score improves. *Key ADK feature: using evaluation as a regression check to prove a change actually helped.*


## Part 0: Set up your environment

In this part you install the ADK and the scoring libraries and point the notebook at your Google Cloud project. Authentication is automatic through the Workbench instance's service account.


Install the [ADK](https://adk.dev/) (with the `eval` extra), the Vertex AI SDK, and the scoring libraries (`rouge-score` for word-overlap scoring, `tabulate` for the result tables). Run the cell below, then continue to the kernel restart.


In [2]:
%pip install --upgrade --quiet \
    "google-adk[eval]==1.34.1" \
    "google-cloud-aiplatform[evaluation]>=1.100.0" \
    "rouge-score>=0.1.2" \
    "tabulate>=0.9.0"

Note: you may need to restart the kernel to use updated packages.


### Restart the kernel

The install above upgrades packages that the base environment already has loaded. Run the next cell to restart the kernel so the new versions load cleanly. The kernel restarts automatically (this is expected) -- wait a few seconds for it to come back, then continue from the cell below it. Do not re-run the install cell.

In [3]:
# Restart the kernel so the updated packages load cleanly. The kernel will
# restart automatically; wait for it to come back, then continue from the next cell.
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

/var/tmp/ipykernel_11982/911708489.py:5: DeprecationWarning: `IPython.Application` is only a re-export of `traitlets.config.application.Application`; import it from traitlets directly. Accessing it here triggers an import of `IPython.core.application`, which is no longer imported when IPython is -- import that module explicitly if you rely on that import happening, in particular if you also rely on other submodules being transitively imported as a side effect.
  IPython.Application.instance().kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

### Set Google Cloud project information

In this lab environment, just run the cell -- it reads your project from the `GOOGLE_CLOUD_PROJECT` environment variable set on the Workbench instance. (Running outside the lab? Set `PROJECT_ID` directly.) The agent's model, `gemini-3.5-flash`, is served in the `global` location, so the cell routes the agent's model calls to `global`; ADK evaluation runs locally in this notebook against that model. Confirm the printed project matches your lab Project ID.

In [1]:
import os
import warnings

# Quiet the experimental-feature notices from ADK and the Vertex SDK.
warnings.filterwarnings("ignore", message=r".*EXPERIMENTAL.*")
warnings.filterwarnings("ignore", message=r".*is experimental.*")

# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
# fmt: on



# Use Vertex AI with the ADK; the agent's model (gemini-3.5-flash) is a global-only
# preview, so route the agent's model calls to 'global'.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

print(f"Project: {PROJECT_ID} | model location: global")


Project: qwiklabs-gcp-03-6abee3031951 | model location: global


### Make the ADK CLI available to shell commands

The `adk` command-line tool installs into the environment's `bin` directory, which the notebook's `!` shell commands do not have on their `PATH`. Add it so the `!adk ...` cells below resolve. The cell has no output; the PATH change takes effect immediately.

In [2]:
import os
import sys

# `adk` installs to the env's bin dir, which isn't on the PATH used by `!` shell
# commands. Prepend it (and the user bin) so the `!adk` cells below resolve.
os.environ["PATH"] = (
    os.path.dirname(sys.executable)
    + os.pathsep + os.path.expanduser("~/.local/bin")
    + os.pathsep + os.environ["PATH"]
)

## Part 1: Build a simple ADK agent

You build a customer service agent for a fictional retailer, Cymbal Home & Garden. An [ADK agent](https://adk.dev/get-started/) is a model, an instruction, and a set of [tools](https://adk.dev/tools-custom/function-tools/). This one has three tools over a small in-memory dataset, so the expected tool calls and answers stay stable run to run (the model's wording still varies):

- `get_purchase_history` returns a customer's past orders and their status.
- `issue_refund` refunds an order and updates its status.
- `lookup_product_info` returns details for a product.

The next three cells write the agent to disk as an ADK package (`customer_service_agent/`), because the ADK evaluation tools load it as a module, then import the same object so you can test it in the notebook.


In [3]:
import os
import sys

os.makedirs("customer_service_agent", exist_ok=True)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())  # so `import customer_service_agent` resolves


Write the agent as an importable module. **Model:** `gemini-3.5-flash`. The instruction has it look up the customer's purchase history and, for a refund, ask for the reason before calling `issue_refund`. The three tools described above are attached here.

In [4]:
%%writefile customer_service_agent/agent.py
import logging
import os
from typing import Dict, List, Any
from google.adk.agents import Agent

import copy
# Mock data stored in memory for the session
DEFAULT_MOCK_DATA = {
    "CUST001": {
        "orders": [
            {"order_id": "ORD-101", "date": "2023-10-15", "items": ["Wireless Headphones"], "total": 120.00, "status": "delivered"},
            {"order_id": "ORD-102", "date": "2023-11-01", "items": ["USB-C Cable", "Phone Case"], "total": 35.00, "status": "shipped"}
        ]
    },
    "CUST002": {
        "orders": [
            {"order_id": "ORD-201", "date": "2023-09-20", "items": ["Smart Watch"], "total": 250.00, "status": "delivered"}
        ]
    }
}

MOCK_DATA = copy.deepcopy(DEFAULT_MOCK_DATA)

def reset_mock_data():
    global MOCK_DATA
    MOCK_DATA = copy.deepcopy(DEFAULT_MOCK_DATA)

def get_purchase_history(customer_id: str) -> Dict[str, Any]:
    """
    Retrieves the purchase history for a given customer, including order status.

    Args:
        customer_id: The unique identifier for the customer.

    Returns:
        A dictionary containing a list of past orders with their current status.
    """
    return MOCK_DATA.get(customer_id, {"orders": [], "message": "No purchase history found for this customer."})

def issue_refund(order_id: str, reason: str) -> Dict[str, Any]:
    """
    Issues a refund for a specific order and updates its status.

    Args:
        order_id: The unique identifier for the order.
        reason: The reason for the refund.

    Returns:
        A dictionary confirming the refund status and updated order information.
    """
    for customer_id, data in MOCK_DATA.items():
        for order in data["orders"]:
            if order["order_id"] == order_id:
                if order["status"] == "refunded":
                    return {
                        "status": "error",
                        "message": f"Order {order_id} has already been refunded."
                    }
                order["status"] = "refunded"
                return {
                    "status": "success",
                    "order_id": order_id,
                    "new_status": "refunded",
                    "refund_amount": f"Full refund of ${order['total']} processed",
                    "message": f"Refund issued for order {order_id} due to: {reason}"
                }
    
    return {
        "status": "error",
        "message": f"Order ID {order_id} not found or not eligible for refund."
    }

def lookup_product_info(product_name: str) -> Dict[str, Any]:
    """
    Looks up details for a specific product.

    Args:
        product_name: The name of the product to look up.

    Returns:
        A dictionary with product details.
    """
    # Mock data
    products = {
        "wireless headphones": {
            "price": 120.00,
            "in_stock": True,
            "description": "Noise-canceling wireless headphones with 20-hour battery life."
        },
        "smart watch": {
            "price": 250.00,
            "in_stock": False,
            "description": "Advanced fitness tracking and notifications."
        },
        "usb-c cable": {
            "price": 15.00,
            "in_stock": True,
            "description": "6ft braided USB-C to USB-C cable."
        }
    }
    
    normalized_name = product_name.lower()
    if normalized_name in products:
        return products[normalized_name]
    else:
        return {"message": "Product not found."}

agent_instruction = """
You are a helpful and efficient retail customer service representative for Cymbal Home & Garden.
Your goal is to assist customers with their purchase history, refunds, and product inquiries.

**Guidelines:**
1.  **Identify the Customer:** If a customer asks about their history or order status, ask for their Customer ID if they haven't provided it.
2.  **Check Order Status:** Use `get_purchase_history` to see the current status of orders (e.g., ordered, shipped, delivered, refunded).
3.  **Handle Refunds:** When a customer wants a refund, ask for the Order ID and the reason for the refund. Use the `issue_refund` tool. If the customer mentions damage, use "damaged" as the reason.
4.  **Product Inquiries:** For product questions, use `lookup_product_info`. If a product is out of stock, inform the customer.
5.  **Be Polite & Professional:** Always use a friendly, professional tone.
6.  **Prioritize Solutions:** Try to resolve the customer's issue quickly using the available tools.

**Available Tools:**
* `get_purchase_history`: Get past orders and their current status for a customer.
* `issue_refund`: Process a refund for an order and update its status.
* `lookup_product_info`: Get details about a product.
"""

agent = Agent(
    model="gemini-3.5-flash",
    name="customer_service_agent",
    instruction=agent_instruction,
    tools=[
        get_purchase_history,
        issue_refund,
        lookup_product_info,
    ],
)

root_agent = agent


Overwriting customer_service_agent/agent.py


Add an `__init__.py` so `customer_service_agent` is an importable package, which lets the `adk eval` CLI load the agent as a module.

In [5]:
%%writefile customer_service_agent/__init__.py
from . import agent  # noqa: F401


Overwriting customer_service_agent/__init__.py


Import the agent back into the notebook and confirm it loaded: the cell prints the agent's name and its three tools.

In [6]:
from customer_service_agent.agent import root_agent as customer_service_agent

print("Agent:", customer_service_agent.name, "| tools:", [t.__name__ for t in customer_service_agent.tools])

Agent: customer_service_agent | tools: ['get_purchase_history', 'issue_refund', 'lookup_product_info']


### Test the agent locally

Run the agent in an in-memory [ADK runner](https://adk.dev/runtime/) and send it one message. Watch it call `get_purchase_history` and then answer. This confirms the agent works before you evaluate it.


In [7]:
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

runner = InMemoryRunner(agent=customer_service_agent, app_name="customer_service_agent")
session = await runner.session_service.create_session(
    app_name="customer_service_agent", user_id="user"
)

async def send(text):
    print(f"USER: {text}")
    async for event in runner.run_async(
        user_id="user",
        session_id=session.id,
        new_message=genai_types.Content(role="user", parts=[genai_types.Part(text=text)]),
    ):
        for part in (event.content.parts if event.content else []):
            if getattr(part, "function_call", None):
                print(f"  [tool call] {part.function_call.name}({dict(part.function_call.args)})")
            elif getattr(part, "function_response", None):
                print(f"  [tool result] {part.function_response.name}")
            elif getattr(part, "text", None):
                print(f"  {event.author}: {part.text.strip()}")

await send("Hi, can you show me the purchase history for customer CUST001?")

USER: Hi, can you show me the purchase history for customer CUST001?
  [tool call] get_purchase_history({'customer_id': 'CUST001'})
  [tool result] get_purchase_history
  customer_service_agent: Here is the purchase history for customer CUST001:

*   **Order ID: ORD-101**
    *   **Date:** October 15, 2023
    *   **Items:** Wireless Headphones
    *   **Total:** $120.00
    *   **Status:** Delivered

*   **Order ID: ORD-102**
    *   **Date:** November 1, 2023
    *   **Items:** USB-C Cable, Phone Case
    *   **Total:** $35.00
    *   **Status:** Shipped

Is there anything else I can assist you with regarding these orders?


## Part 2: Evaluate the agent locally with ADK

ADK ships its own [evaluation framework](https://adk.dev/evaluate/) that runs on your machine and in CI, so you catch regressions as you build. You capture the agent's expected behavior as data, then grade an actual run against it.

ADK stores expected behavior in two file types that share one schema and differ only in how many sessions they hold. A **test file** (`*.test.json`) holds a single session for a fast unit check (this lab does not use one). An **eval set** (`*.evalset.json`) holds several multi-turn sessions for broader coverage. Each file holds `eval_cases`; each case is a `conversation` of invocations; each invocation pairs the `user_content` with the reference `final_response` and records the expected tool calls under `intermediate_data`.


The eval set for this lab (2 eval cases) is included with the lab files at `customer_service_agent/cs_eval_set.evalset.json`. Here is its purchase-history invocation, trimmed:

```json
{
  "user_content": {
    "parts": [{ "text": "Can you show me the purchase history for customer CUST001?" }],
    "role": "user"
  },
  "final_response": {
    "parts": [{ "text": "Customer CUST001 has two orders: ORD-101 (Wireless Headphones, ...) and ORD-102 (...)." }],
    "role": "model"
  },
  "intermediate_data": {
    "invocation_events": [
      {
        "content": {
          "role": "model",
          "parts": [{ "function_call": { "name": "get_purchase_history", "args": { "customer_id": "CUST001" } } }]
        }
      }
    ]
  }
}
```

The expected tool calls live under `intermediate_data.invocation_events` as `function_call` entries; the file also records each tool's response. This is what the tool-trajectory metric grades against. Open the file in the JupyterLab file browser to see both cases in full.

### Run the reference metrics

Two **[reference metrics](https://adk.dev/evaluate/criteria/)** are computed entirely on your machine, so the inner loop stays fast:

- **`tool_trajectory_avg_score`** compares the agent's actual tool calls to the expected ones (tool name and arguments). It defaults to an exact match at threshold 1.0.
- **`response_match_score`** compares the final response to the reference answer with ROUGE-1 (word overlap). It defaults to 0.8.

You set thresholds in a config file, then run the `adk eval` CLI: it takes the agent package directory, the eval-set file, and a `--config_file_path`. The first cell writes the config; the second runs the eval. The agent's own model calls go to Vertex AI; the metric computation is local and adds no cost.


In [8]:
%%writefile customer_service_agent/eval_config.reference.json
{
  "criteria": {
    "tool_trajectory_avg_score": 1.0,
    "response_match_score": 0.8
  }
}


Writing customer_service_agent/eval_config.reference.json


In [9]:
!adk eval customer_service_agent customer_service_agent/cs_eval_set.evalset.json \
    --config_file_path customer_service_agent/eval_config.reference.json \
    --print_detailed_results --log_level=CRITICAL

/opt/micromamba/lib/python3.12/site-packages/google/adk/dependencies/vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions wit

**Readable summary** *(added afterward, not part of the original run output)*

| Case | Metric | Score | Result |
|---|---|:---:|:---:|
| `refund_damaged_item` | `tool_trajectory_avg_score` | 1.0 | ✅ |
| `refund_damaged_item` | `response_match_score` | 0.57 | ❌ |
| `purchase_history_lookup` | `tool_trajectory_avg_score` | 1.0 | ✅ |
| `purchase_history_lookup` | `response_match_score` | 0.75 | ❌ |

Both cases fail only on `response_match_score` (ROUGE-1, literal word overlap) — the agent paraphrased instead of matching the reference wording exactly.


> **Note:** the agent and the model are nondeterministic, so an exact tool or
> wording match can score below threshold on some runs. That is the reference
> metrics doing their job, and it is why you add a semantic judge metric next.


### Add an LLM-judge metric

Reference metrics check exact tool calls and wording. **[LLM-judge metrics](https://adk.dev/evaluate/criteria/)** check whether the answer *means* the right thing. `final_response_match_v2` asks a judge model whether the response matches the reference semantically, which is robust to wording. Judge metrics call a judge model on Vertex AI, so they are billable and need an explicit threshold. Write the config, then run the eval. In the config, `num_samples: 5` calls the judge five times per response for a steadier score. The run takes a minute or two -- and expect it to **pass**: the judge accepts the same answers the exact word-match scored below threshold.


In [10]:
%%writefile customer_service_agent/eval_config.judge.json
{
  "criteria": {
    "final_response_match_v2": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-3.5-flash",
        "num_samples": 5
      }
    }
  }
}


Writing customer_service_agent/eval_config.judge.json


In [11]:
!adk eval customer_service_agent customer_service_agent/cs_eval_set.evalset.json \
    --config_file_path customer_service_agent/eval_config.judge.json \
    --print_detailed_results --log_level=CRITICAL

/opt/micromamba/lib/python3.12/site-packages/google/adk/dependencies/vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions wit

**Readable summary** *(added afterward, not part of the original run output)*

| Case | Metric | Score | Result |
|---|---|:---:|:---:|
| `refund_damaged_item` | `final_response_match_v2` | 1.0 | ✅ |
| `purchase_history_lookup` | `final_response_match_v2` | 1.0 | ✅ |

Same responses `response_match_score` rejected above — the LLM judge recognizes them as correct.


### Grade with a rubric

`rubric_based_final_response_quality_v1` scores the response against [rubrics](https://adk.dev/evaluate/criteria/) you write, so you grade exactly the qualities you care about (here, conciseness and completeness). Like other judge metrics, it calls a judge model on Vertex AI. Write the config, then run the eval; it takes a minute or two.


In [12]:
%%writefile customer_service_agent/eval_config.rubric.json
{
  "criteria": {
    "rubric_based_final_response_quality_v1": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-3.5-flash",
        "num_samples": 5
      },
      "rubrics": [
        {
          "rubric_id": "conciseness",
          "rubric_content": {
            "text_property": "The response is direct and concise, without unnecessary detail."
          }
        },
        {
          "rubric_id": "completeness",
          "rubric_content": {
            "text_property": "The response fully answers the customer's request, including the relevant order details."
          }
        }
      ]
    }
  }
}


Writing customer_service_agent/eval_config.rubric.json


In [13]:
!adk eval customer_service_agent customer_service_agent/cs_eval_set.evalset.json \
    --config_file_path customer_service_agent/eval_config.rubric.json \
    --print_detailed_results --log_level=CRITICAL

/opt/micromamba/lib/python3.12/site-packages/google/adk/dependencies/vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions wit

**Readable summary** *(added afterward, not part of the original run output)*

| Case | Rubric | Score |
|---|---|:---:|
| `purchase_history_lookup` | conciseness | 1.0 |
| `purchase_history_lookup` | completeness | 1.0 |
| `refund_damaged_item` | conciseness | 1.0 |
| `refund_damaged_item` | completeness | 0.0 |

**Overall**: `purchase_history_lookup` ✅ pass (1.0) · `refund_damaged_item` ❌ fail (0.5) — dragged down entirely by `completeness`. See the README's "wrong judge reason" note: the judge marked real, tool-sourced data as "hallucinated" on this case's 2nd turn.


> **Note:** expect a FAILED result here. Check the per-rubric scores in the output:
> `conciseness` passes, but `completeness` scores low on the refund case. Why? The
> completeness rubric asks for "the relevant order details," but the agent's first
> refund turn asks the customer for the reason instead. That is the correct behavior --
> the rubric text just doesn't account for it. The lesson: a judge grades exactly the
> rubric you wrote, so word rubrics to match the behavior you actually want, and read
> the per-rubric scores rather than only the overall pass/fail.

### Other ADK criteria

ADK supports more judge-based [criteria](https://adk.dev/evaluate/criteria/) you can add to the `criteria` block. Each needs an explicit threshold and calls a judge model (billable):

| Criterion | What it grades | Example config value |
|---|---|---|
| `rubric_based_final_response_quality_v1` | response against rubrics you write | `{ "threshold": 0.8, "judge_model_options": {...}, "rubrics": [...] }` |
| `rubric_based_tool_use_quality_v1` | tool use against rubrics | `{ "threshold": 0.8, "judge_model_options": {...}, "rubrics": [...] }` |
| `hallucinations_v1` | claims grounded in context/tool output | `{ "threshold": 0.5, "evaluate_intermediate_nl_responses": false }` |
| `safety_v1` | response safety | `0.8` |


## Part 3: Evaluate with ADK user simulation

The cases so far were fixed scripts. ADK can also evaluate your agent against a *[simulated user](https://adk.dev/evaluate/user-sim/)* that improvises a multi-turn conversation toward a goal. You give it a `starting_prompt` and a `conversation_plan`, and a model plays the customer.

This is ADK's own dynamic evaluation. (The managed platform has its own User Simulator, which you will see in Lab B.)


### Create the config and scenario files

For the user-simulation eval you write four small files. First, the **session input** below sets the `app_name` and `user_id` for the simulated session.

In [14]:
%%writefile customer_service_agent/session_input.json
{
  "app_name": "customer_service_agent",
  "user_id": "user"
}


Writing customer_service_agent/session_input.json


The **dry-run eval config**: no metrics (`criteria: {}`), just the **user simulator** (`model: gemini-3.5-flash`; `max_allowed_invocations: 20` caps the total turns). Run this first to generate the conversation without scoring.

In [15]:
%%writefile customer_service_agent/eval_config_without_metrics.json
{
  "criteria": {},
  "user_simulator_config": {
    "model": "gemini-3.5-flash",
    "model_configuration": {
      "thinking_config": {
        "include_thoughts": true,
        "thinking_budget": 10240
      }
    },
    "max_allowed_invocations": 20
  }
}


Writing customer_service_agent/eval_config_without_metrics.json


The **scored eval config**: the same user simulator plus two judge metrics -- **`hallucinations_v1`** (threshold 0.5) and **`safety_v1`** (threshold 0.8).

In [16]:
%%writefile customer_service_agent/eval_config_with_metrics.json
{
  "criteria": {
    "hallucinations_v1": {
      "threshold": 0.5
    },
    "safety_v1": {
      "threshold": 0.8
    }
  },
  "user_simulator_config": {
    "model": "gemini-3.5-flash",
    "model_configuration": {
      "thinking_config": {
        "include_thoughts": true,
        "thinking_budget": 10240
      }
    },
    "max_allowed_invocations": 20
  }
}


Writing customer_service_agent/eval_config_with_metrics.json


The **conversation scenario** that drives the simulated user: a `starting_prompt` and a `conversation_plan` (customer CUST001 requests a refund on the damaged headphones, order ORD-101).

In [17]:
%%writefile customer_service_agent/conversation_scenarios.json
{
  "scenarios": [
    {
      "starting_prompt": "Hi, I need help with a recent order.",
      "conversation_plan": "You are customer CUST001. First ask the agent to look up your purchase history. Then ask for a refund on the wireless headphones (order ORD-101) because it arrived damaged. Give the order ID and the reason when the agent asks."
    }
  ]
}


Writing customer_service_agent/conversation_scenarios.json


### Build the eval set from the scenario

`adk eval_set create` makes an empty eval set, and `adk eval_set add_eval_case`
turns your scenario into a runnable case.

In [18]:
print("Creating the eval set...", flush=True)
!adk eval_set create customer_service_agent cs_user_sim --log_level=CRITICAL

print("Adding the conversation scenario as an eval case...", flush=True)
!adk eval_set add_eval_case customer_service_agent cs_user_sim \
    --scenarios_file customer_service_agent/conversation_scenarios.json \
    --session_input_file customer_service_agent/session_input.json \
    --log_level=CRITICAL

Creating the eval set...
Eval set 'cs_user_sim' created for app 'customer_service_agent'.
Adding the conversation scenario as an eval case...
Eval case '5e613b45' added to eval set 'cs_user_sim'.


### See the generated eval case

The new eval set is saved locally at `customer_service_agent/cs_user_sim.evalset.json`.
Each case stores the *scenario* the simulator will follow (its `starting_prompt` and
`conversation_plan`); the actual turn-by-turn conversation is generated when you run
the eval below. Read the first case to confirm it captured your scenario.


In [19]:
import json

with open("customer_service_agent/cs_user_sim.evalset.json") as f:
    user_sim_set = json.load(f)

print("Stored at: customer_service_agent/cs_user_sim.evalset.json")
print("Eval cases:", len(user_sim_set["eval_cases"]), "\n")

first = user_sim_set["eval_cases"][0]
print("First case id:", first.get("eval_id"))
scenario = first.get("conversation_scenario")
print(json.dumps(scenario if scenario else first, indent=2))


Stored at: customer_service_agent/cs_user_sim.evalset.json
Eval cases: 1 

First case id: 5e613b45
{
  "starting_prompt": "Hi, I need help with a recent order.",
  "conversation_plan": "You are customer CUST001. First ask the agent to look up your purchase history. Then ask for a refund on the wireless headphones (order ORD-101) because it arrived damaged. Give the order ID and the reason when the agent asks."
}


### Dry run: check the conversation, no scoring

Run the simulation with the no-metrics config first to read the dialogue and
confirm the simulated user follows the plan. `Overall Eval Status: NOT_EVALUATED`
is expected, because no metrics were scored. The summary may also show `Tests failed: 1` -- with no metrics scored, that is a summary-table artifact, not a failure.

In [20]:
!adk eval customer_service_agent cs_user_sim \
    --config_file_path customer_service_agent/eval_config_without_metrics.json \
    --print_detailed_results --log_level=CRITICAL

/opt/micromamba/lib/python3.12/site-packages/google/adk/dependencies/vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions wit

**Readable summary** *(added afterward, not part of the original run output)*

Dry run, no metrics configured (`criteria: {}`) — `Overall Eval Status: NOT_EVALUATED` is expected. This run only confirms the simulated conversation followed the scenario; nothing gets scored yet.


### Scored run: hallucinations and safety

Now run the same simulation with metrics. This scores each turn for `hallucinations_v1` (claims grounded in tool output) and `safety_v1`. With user simulation, ADK supports only these two [criteria](https://adk.dev/evaluate/criteria/), because there is no fixed expected answer to match against. These call a judge model on Vertex AI, so the run is billable and takes a couple of minutes.


In [21]:
!adk eval customer_service_agent \
    --config_file_path customer_service_agent/eval_config_with_metrics.json \
    cs_user_sim \
    --print_detailed_results --log_level=CRITICAL

/opt/micromamba/lib/python3.12/site-packages/google/adk/dependencies/vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions wit

**Readable summary** *(added afterward, not part of the original run output)*

| Metric | Score | Result |
|---|:---:|:---:|
| `hallucinations_v1` | 0.97 | ✅ |
| `safety_v1` | 0.0 | ❌ |

Same 3-turn conversation, zero unsafe content — see the README for why `safety_v1`'s flat 0.0 looks like a bug, not a real verdict.


## Part 4: Optimize and verify

Evaluation is only useful if it drives a fix. Here you see the [build-time loop](https://adk.dev/evaluate/) end to end.

Suppose an earlier version of the agent (**v1**) had a weaker refund instruction: it issued the refund immediately instead of asking the customer for the reason first. You evaluate v1 against the refund case with the tool-trajectory metric, watch it fail, then confirm the current agent (**v2**, the one you built) passes the same check. That score change is the proof your instruction fix worked.

The next cells write a v1 agent (with the gap) and a tool-trajectory eval config; the refund eval sets ship with the lab files. Then you run both versions and compare.


In [22]:
import os
os.makedirs("customer_service_agent_v1", exist_ok=True)

Write **v1** of the agent. Same **model** (`gemini-3.5-flash`) and tools, but a weaker refund instruction: it issues the refund immediately instead of asking the customer for the reason first. This is the seeded gap you catch with evaluation.

In [23]:
%%writefile customer_service_agent_v1/agent.py
import logging
import os
from typing import Dict, List, Any
from google.adk.agents import Agent

import copy
# Mock data stored in memory for the session
DEFAULT_MOCK_DATA = {
    "CUST001": {
        "orders": [
            {"order_id": "ORD-101", "date": "2023-10-15", "items": ["Wireless Headphones"], "total": 120.00, "status": "delivered"},
            {"order_id": "ORD-102", "date": "2023-11-01", "items": ["USB-C Cable", "Phone Case"], "total": 35.00, "status": "shipped"}
        ]
    },
    "CUST002": {
        "orders": [
            {"order_id": "ORD-201", "date": "2023-09-20", "items": ["Smart Watch"], "total": 250.00, "status": "delivered"}
        ]
    }
}

MOCK_DATA = copy.deepcopy(DEFAULT_MOCK_DATA)

def reset_mock_data():
    global MOCK_DATA
    MOCK_DATA = copy.deepcopy(DEFAULT_MOCK_DATA)

def get_purchase_history(customer_id: str) -> Dict[str, Any]:
    """
    Retrieves the purchase history for a given customer, including order status.

    Args:
        customer_id: The unique identifier for the customer.

    Returns:
        A dictionary containing a list of past orders with their current status.
    """
    return MOCK_DATA.get(customer_id, {"orders": [], "message": "No purchase history found for this customer."})

def issue_refund(order_id: str, reason: str) -> Dict[str, Any]:
    """
    Issues a refund for a specific order and updates its status.

    Args:
        order_id: The unique identifier for the order.
        reason: The reason for the refund.

    Returns:
        A dictionary confirming the refund status and updated order information.
    """
    for customer_id, data in MOCK_DATA.items():
        for order in data["orders"]:
            if order["order_id"] == order_id:
                if order["status"] == "refunded":
                    return {
                        "status": "error",
                        "message": f"Order {order_id} has already been refunded."
                    }
                order["status"] = "refunded"
                return {
                    "status": "success",
                    "order_id": order_id,
                    "new_status": "refunded",
                    "refund_amount": f"Full refund of ${order['total']} processed",
                    "message": f"Refund issued for order {order_id} due to: {reason}"
                }
    
    return {
        "status": "error",
        "message": f"Order ID {order_id} not found or not eligible for refund."
    }

def lookup_product_info(product_name: str) -> Dict[str, Any]:
    """
    Looks up details for a specific product.

    Args:
        product_name: The name of the product to look up.

    Returns:
        A dictionary with product details.
    """
    # Mock data
    products = {
        "wireless headphones": {
            "price": 120.00,
            "in_stock": True,
            "description": "Noise-canceling wireless headphones with 20-hour battery life."
        },
        "smart watch": {
            "price": 250.00,
            "in_stock": False,
            "description": "Advanced fitness tracking and notifications."
        },
        "usb-c cable": {
            "price": 15.00,
            "in_stock": True,
            "description": "6ft braided USB-C to USB-C cable."
        }
    }
    
    normalized_name = product_name.lower()
    if normalized_name in products:
        return products[normalized_name]
    else:
        return {"message": "Product not found."}

agent_instruction = """
You are a helpful and efficient retail customer service representative for Cymbal Home & Garden.
Your goal is to assist customers with their purchase history, refunds, and product inquiries.

**Guidelines:**
1.  **Identify the Customer:** If a customer asks about their history or order status, ask for their Customer ID if they haven't provided it.
2.  **Check Order Status:** Use `get_purchase_history` to see the current status of orders (e.g., ordered, shipped, delivered, refunded).
3.  **Handle Refunds:** When a customer wants a refund, immediately call the `issue_refund` tool using the order ID. Do not ask the customer for a reason.
4.  **Product Inquiries:** For product questions, use `lookup_product_info`. If a product is out of stock, inform the customer.
5.  **Be Polite & Professional:** Always use a friendly, professional tone.
6.  **Prioritize Solutions:** Try to resolve the customer's issue quickly using the available tools.

**Available Tools:**
* `get_purchase_history`: Get past orders and their current status for a customer.
* `issue_refund`: Process a refund for an order and update its status.
* `lookup_product_info`: Get details about a product.
"""

agent = Agent(
    model="gemini-3.5-flash",
    name="customer_service_agent_v1",
    instruction=agent_instruction,
    tools=[
        get_purchase_history,
        issue_refund,
        lookup_product_info,
    ],
)

root_agent = agent


Writing customer_service_agent_v1/agent.py


Add the package `__init__.py` for the v1 agent.

In [26]:
%%writefile customer_service_agent_v1/__init__.py
from . import agent  # noqa: F401


Overwriting customer_service_agent_v1/__init__.py


The refund eval set is included with the lab for both agent versions (`customer_service_agent_v1/cs_refund.evalset.json` and `customer_service_agent/cs_refund.evalset.json`). It holds the same refund test cases; you run it against the gapped v1 agent and the fixed agent to compare their tool trajectories.

The **trajectory-only eval config**: scores just **`tool_trajectory_avg_score`** (threshold 1.0), so you compare v1 and the fixed agent purely on whether they make the right tool calls.

In [27]:
%%writefile customer_service_agent/eval_config.trajectory.json
{
  "criteria": {
    "tool_trajectory_avg_score": 1.0
  }
}


Overwriting customer_service_agent/eval_config.trajectory.json


### Before: v1 (issues the refund without asking)

v1 should call `issue_refund` on the first turn, which does not match the expected
trajectory (ask first, refund second), so `tool_trajectory_avg_score` fails.

In [28]:
!adk eval customer_service_agent_v1 customer_service_agent_v1/cs_refund.evalset.json \
    --config_file_path customer_service_agent/eval_config.trajectory.json \
    --print_detailed_results --log_level=CRITICAL

/opt/micromamba/lib/python3.12/site-packages/google/adk/dependencies/vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions wit

**Readable summary** *(added afterward, not part of the original run output)*

| Agent | Metric | Score | Result |
|---|---|:---:|:---:|
| `customer_service_agent_v1` (seeded bug) | `tool_trajectory_avg_score` | 0.0 | ❌ |


### After: v2 (asks for the reason first)

The current agent asks for the reason on the first turn and issues the refund on
the second, matching the expected trajectory, so it passes.

In [29]:
!adk eval customer_service_agent customer_service_agent/cs_refund.evalset.json \
    --config_file_path customer_service_agent/eval_config.trajectory.json \
    --print_detailed_results --log_level=CRITICAL

/opt/micromamba/lib/python3.12/site-packages/google/adk/dependencies/vertexai.py:19: UserWarning: The `vertexai.preview.rag` module is deprecated and will be removed in a future version. Please migrate to the `agentplatform` client. For example:

    import agentplatform

    client = agentplatform.Client(project="your-project", location="us-central1")
    client.rag.create_corpus(...)

  from vertexai.preview import rag
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/micromamba/lib/python3.12/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions wit

**Readable summary** *(added afterward, not part of the original run output)*

| Agent | Metric | Score | Result |
|---|---|:---:|:---:|
| `customer_service_agent` (current) | `tool_trajectory_avg_score` | 1.0 | ✅ |

Same eval set as the cell above — one weaker instruction line in `v1` is the whole difference between 0.0 and 1.0.


A one-line difference in the instruction turned a failing trajectory into a passing one. That is the build-time loop ADK is designed for: write expected
behavior, measure against it, fix, and re-measure.

## Recap

You evaluated an ADK agent without leaving the notebook, using ADK's own framework:

- **Reference metrics** (`tool_trajectory_avg_score`, `response_match_score`) for fast, local, exact checks.
- **LLM-judge metrics** (`final_response_match_v2`, `rubric_based_final_response_quality_v1`) for semantic and rubric quality.
- **User simulation** to generate a dynamic multi-turn conversation, scored with `hallucinations_v1` and `safety_v1`.
- An **optimize and verify** loop: a weaker instruction failed the refund trajectory, and the fixed instruction passed it.

ADK evaluation is the fast inner loop you run before an agent reaches the platform. Evaluating a *deployed* agent with the Gemini Enterprise Agent Platform's managed tools is **Lab B**, a separate notebook.

### Learn more

- [ADK documentation](https://adk.dev/)
- [Evaluate agents](https://adk.dev/evaluate/)
- [Evaluation criteria and metrics](https://adk.dev/evaluate/criteria/)
- [User simulation](https://adk.dev/evaluate/user-sim/)
- [Custom metrics](https://adk.dev/evaluate/custom_metrics/)
